<a href="https://colab.research.google.com/github/ShahaanK/Relic/blob/main/Relic_Synthetic_Chat_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import random
from datetime import datetime, timedelta
from openai import OpenAI
from google.colab import userdata
import time


In [ ]:

# ==========================================
# CONFIGURATION & API SETUP
# ==========================================
# If using NVIDIA NIM, replace these with your NIM endpoint and key.
# Example NIM base_url: "https://integrate.api.nvidia.com/v1"
# Example NIM model: "nvidia/nemotron-4-340b-instruct"

client = OpenAI(
    api_key=userdata.get('NVI-KEY'), # Set to "" if using a local mock/testing environment
    base_url="https://integrate.api.nvidia.com/v1" # Change to NVIDIA NIM URL if using NVIDIA
)
MODEL_NAME = "meta/llama-3.1-8b-instruct" # Or your desired model

# ==========================================
# STEP 1: DEFINE PERSONAS
# ==========================================
TARGET_PERSONA = {
    "name": "Robert",
    "system_prompt": """You are Robert, a 62-year-old retired accountant. You are texting your 30-year-old adult child, Alex.
Your traits: You care about lawn maintenance and car maintenance.
Your texting style: You use ellipses (...) frequently. You sometimes capitalize random Words. You end conversations with 'Love, Dad'. You are clumsy with technology."""
}

USER_PERSONA = {
    "name": "Alex",
    "system_prompt": """You are Alex, a 30-year-old. You are texting your father, Robert.
Your texting style: Casual, modern, uses lowercase letters, uses typical modern emojis (😂, 💀). You love your dad but are sometimes mildly exasperated by his dad-isms."""
}

# ==========================================
# HELPER FUNCTIONS (STEP 4: HUMAN NOISE)
# ==========================================
def introduce_typo(text):
    """Randomly swaps two adjacent characters to simulate a typing error."""
    if len(text) < 5 or random.random() > 0.15: # 15% chance of a typo
        return text
    idx = random.randint(0, len(text) - 2)
    text_list = list(text)
    text_list[idx], text_list[idx+1] = text_list[idx+1], text_list[idx]
    return "".join(text_list)

# ==========================================
# CORE SIMULATION LOGIC
# ==========================================
def generate_timeline(num_events=5):
    """Step 2: Generate events with probabilistic severity to dictate conversation length."""
    prompt = f"""Generate {num_events} events that a 62-year-old father and his 30-year-old child would text about over a few months.
Mix mundane tasks (taking out trash) with serious situations (hospital visits).
CRITICAL: Output ONLY a valid JSON object with the key 'events', containing a list of objects. Each object MUST have:
- "description": (string, 1 sentence)
- "severity": (integer 1-10, where 1 is a trivial chore, 10 is highly serious/urgent)"""

    try:
        time.sleep(2)
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            response_format={"type": "json_object"}
        )

        raw_content = response.choices[0].message.content.strip()

        # Clean JSON formatting if the model wrapped it in markdown
        if raw_content.startswith("```json"):
            raw_content = raw_content[7:]
        if raw_content.startswith("```"):
            raw_content = raw_content[3:]
        if raw_content.endswith("```"):
            raw_content = raw_content[:-3]

        parsed_data = json.loads(raw_content)
        events = parsed_data.get("events", [])
        return events[:num_events]

    except Exception as e:
        print(f"Error generating timeline: {e}. Using fallbacks.")
        return [
            {"description": "Asking if Alex checked their tire pressure.", "severity": 2},
            {"description": "Discussing Sunday dinner plans.", "severity": 4},
            {"description": "Robert had a minor fall and is at the urgent care.", "severity": 8}
        ]

def generate_agent_reply(persona_prompt, scenario, conversation_history, sender_name):
    """Step 3: Call the LLM to get a single text message reply."""
    messages = [
        {"role": "system", "content": persona_prompt}
    ]

    # Package the history into a strict text block inside the user prompt
    # rather than confusing the API with fake role assignments.
    prompt = f"Topic/Scenario: {scenario}\n\n--- RECENT MESSAGES ---\n"
    if not conversation_history:
        prompt += "(No messages yet. You are starting the conversation.)\n"
    else:
        for msg in conversation_history[-6:]:
            prompt += f"{msg['sender']}: {msg['message']}\n"

    prompt += f"-----------------------\n\n"
    prompt += f"Write the next text message as '{sender_name}'.\n"
    prompt += "CRITICAL: You MUST respond with a valid JSON object containing exactly one key 'message'. Do NOT include any explanations, greetings, or meta-talk. Example:\n"
    prompt += "{\"message\": \"your text here\"}"

    messages.append({"role": "user", "content": prompt})

    try:
        time.sleep(2)
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.7,
            max_tokens=150 # Increased to prevent cutoff mid-JSON
        )
        raw_reply = response.choices[0].message.content.strip()

        # Clean JSON formatting if the model wrapped it in markdown
        if raw_reply.startswith("```json"):
            raw_reply = raw_reply[7:]
        if raw_reply.startswith("```"):
            raw_reply = raw_reply[3:]
        if raw_reply.endswith("```"):
            raw_reply = raw_reply[:-3]

        raw_reply = raw_reply.strip()

        try:
            # Parse the JSON response
            reply_dict = json.loads(raw_reply)
            reply = reply_dict.get("message", raw_reply)
        except json.JSONDecodeError:
            # Fallback if it didn't output JSON
            reply = raw_reply

        # Strip out any accidentally generated meta-tags, prefixes, or quotes
        prefixes_to_strip = [f"{sender_name}:", "Alex:", "Dad:", "Robert:", "\"", "'"]
        for prefix in prefixes_to_strip:
            if reply.lower().startswith(prefix.lower()):
                reply = reply[len(prefix):].strip()
        if reply.endswith("\"") or reply.endswith("'"):
            reply = reply[:-1].strip()

        return reply
    except Exception as e:
        print(f"API Error: {e}")
        return "..."

def run_simulation(output_file="historical_chat_log.jsonl"):
    """Main function to run the multi-agent loop and save to JSONL."""
    print("Starting simulation...")
    events = generate_timeline(num_events=5) # Start small for testing
    print(f"Generated {len(events)} events.")

    chat_log = []
    current_time = datetime(2023, 1, 1, 9, 0) # Start date

    for event_data in events:
        # Extract the description and severity from the JSON object
        if isinstance(event_data, str): # Fallback safety
            event = event_data
            severity = 3
        else:
            event = event_data.get("description", "Unknown event")
            severity = event_data.get("severity", 3)

        print(f"\nSimulating Event (Severity {severity}/10): {event}")
        # Advance time by a few days for each new event
        current_time += timedelta(days=random.randint(2, 10), hours=random.randint(0, 12))

        # Determine who starts the conversation for this event
        turn = TARGET_PERSONA if random.choice([True, False]) else USER_PERSONA

        # Probabilistic turn count based on severity
        if severity <= 3:
            num_turns = random.randint(2, 6)   # Quick check-ins, mundane chores
        elif severity <= 7:
            num_turns = random.randint(7, 15)  # Planning, catching up, mild arguments
        else:
            num_turns = random.randint(16, 40) # Serious topics, emergencies, deep conversations

        for _ in range(num_turns):
            other_persona = USER_PERSONA if turn == TARGET_PERSONA else TARGET_PERSONA

            # Generate the message (Now passing turn['name'])
            raw_message = generate_agent_reply(turn['system_prompt'], event, chat_log, turn['name'])
            final_message = introduce_typo(raw_message)

            # Record the message
            msg_record = {
                "timestamp": current_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sender": turn['name'],
                "message": final_message
            }
            chat_log.append(msg_record)
            print(f"[{msg_record['timestamp']}] {msg_record['sender']}: {msg_record['message']}")

            # Advance time by a few minutes for the reply
            current_time += timedelta(minutes=random.randint(1, 45))

            # Swap turns (unless they double text - 10% chance)
            if random.random() > 0.10:
                turn = other_persona
            else:
                current_time += timedelta(seconds=random.randint(15, 60)) # Quick follow up

    # Step 5: Save to JSONL
    print(f"\nSaving {len(chat_log)} messages to {output_file}...")
    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in chat_log:
            f.write(json.dumps(entry) + '\n')
    print("Simulation complete!")

if __name__ == "__main__":
    # Ensure you have your API key set at the top before running!
    run_simulation()

Starting simulation...
Generated 5 events.

Simulating Event (Severity 1/10): I took out the trash today.
[2023-01-04T20:00:00Z] Alex: just got done with it, dad. 👍
[2023-01-04T20:26:00Z] Robert: glad you remembered to take out the trash... good job on that... now don't forget to mow the lawn this weekend... it's getting a bit long...
[2023-01-04T21:06:00Z] Alex: dude, i'm not a kid anymore, can't i just take care of it without reminders? 😂

Simulating Event (Severity 4/10): Your mom had a minor accident at home and we're okay.
[2023-01-14T23:14:00Z] Robert: sorry alex... just trying to help... lawn care is important... dont wanna be that house on the block with the overgrown grass...
[2023-01-14T23:41:00Z] Alex: dude, we're fine, mom's okay ,can we focus on that instead of the lawn? 🙄
[2023-01-14T23:56:00Z] Robert: oh right... mom... sorry about that... how is she doing...?
[2023-01-15T00:00:00Z] Alex: she's a bit shaken up but otherwise okay, just a minor cut and some bruises. we're 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
